In [1]:
import sys
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# =====================================================================
# 1. GENERATE HIGH-ALPHA MULTI-YEAR STOCK MARKET TRACKING DATABASE
# =====================================================================
print("Generating multi-year synthetic stock market database (2022 - 2026)...")

# Define total multi-year tracking timeline matching our validation gates
start_date = datetime(2022, 1, 1)
end_date = datetime(2026, 7, 1)  

# Generate everyday dates and filter out weekends for active market timelines
total_days = (end_date - start_date).days
market_dates = [start_date + timedelta(days=i) for i in range(total_days)]
market_dates = [d for d in market_dates if d.weekday() < 5]

# Define tickers across institutional target sectors
sector_mapping = {
    'Banking': ['HDFC', 'ICICI', 'SBI'],
    'IT': ['INFY', 'TCS', 'WIPRO'],
    'Pharma': ['CIPLA', 'LUPIN', 'REDDY', 'SUN']
}

stock_rows = []
np.random.seed(42)

for sector, tickers in sector_mapping.items():
    for ticker in tickers:
        # Assign baseline scaling constants per sector
        if sector == 'IT':
            base_close = 1500.0
        elif sector == 'Pharma':
            base_close = 800.0
        else:
            base_close = 500.0
            
        for date in market_dates:
            # Daily random walk with an embedded structural market drift
            pct_change = np.random.normal(0.0003, 0.012)
            close_price = base_close * (1 + pct_change)
            
            # Intraday spreads to build flawless high-alpha OHLC bars
            intraday_volatility = np.random.uniform(0.005, 0.018)
            open_price = close_price * (1 + np.random.normal(0, 0.002))
            high_price = max(open_price, close_price) * (1 + intraday_volatility)
            low_price = min(open_price, close_price) * (1 - intraday_volatility)
            
            stock_rows.append({
                'date': date.strftime('%Y-%m-%d'),
                'ticker': ticker,
                'sector': sector,
                'open': open_price,
                'high': high_price,
                'low': low_price,
                'close': close_price
            })
            base_close = close_price

raw_market_data = pd.DataFrame(stock_rows)
print(f"-> Stock database created. Total rows: {raw_market_data.shape[0]}")
print(f"-> Target dates spanned: {raw_market_data['date'].min()} to {raw_market_data['date'].max()}")

# =====================================================================
# 2. GENERATE COMPREHENSIVE MACRO-FUNDAMENTAL TIME-SERIES
# =====================================================================
print("\nGenerating tracking macro-fundamental database...")
macro_rows = []

current_rate = 6.50
current_fx = 83.50

for date in market_dates:
    # Simulate historical adjustments over time (e.g., policy updates)
    if date.strftime('%Y-%m-%d') == '2024-06-10':
        current_rate = 6.75
    elif date.strftime('%Y-%m-%d') == '2026-02-15':
        current_rate = 6.25 
        
    current_fx += np.random.normal(0.002, 0.08)
    
    macro_rows.append({
        'date': date.strftime('%Y-%m-%d'),
        'repo_rate': current_rate,
        'usd_inr': current_fx
    })

macro_dataframe = pd.DataFrame(macro_rows)
print(f"-> Macro database successfully created. Shape: {macro_dataframe.shape}")

Generating multi-year synthetic stock market database (2022 - 2026)...
-> Stock database created. Total rows: 11720
-> Target dates spanned: 2022-01-03 to 2026-06-30

Generating tracking macro-fundamental database...
-> Macro database successfully created. Shape: (1172, 3)


In [2]:


# 1. Force Python to find the parent project root containing the 'src' directory
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. Now import your upgraded engine explicitly
from src.data_engine import SectorDataEngine

# Initialize data engine
data_engine = SectorDataEngine()
processed_sector_data = {}
all_sectors = raw_market_data['sector'].unique()

print("Processing indicators with macro overlays sector by sector...")
for sector in all_sectors:
    # Pass both datasets down into our upgraded engine method
    sector_df = data_engine.get_sector_data(raw_market_data, macro_dataframe, sector)
    processed_sector_data[sector] = sector_df
    print(f"-> {sector} sector processed. Shape: {sector_df.shape} | Memory optimized.")

Processing indicators with macro overlays sector by sector...
-> Banking sector processed. Shape: (3366, 34) | Memory optimized.
-> IT sector processed. Shape: (3366, 34) | Memory optimized.
-> Pharma sector processed. Shape: (4488, 34) | Memory optimized.


In [3]:
import sys
import os
import pandas as pd

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path: sys.path.append(project_root)

from src.model_engine import SectorModelEngine

features = [
    'close', 'sma_10', 'sma_50', 'rsi_14', 'repo_rate', 'usd_inr', 'usd_inr_momentum',
    'sma_20', 'ema_12', 'ema_26', 'macd', 'macd_signal', 'macd_hist',
    'bollinger_high', 'bollinger_low', 'bollinger_width', 'atr_14', 'daily_return_volatility',
    'stochastic_k', 'stochastic_d', 'williams_r', 'cci_20', 'rate_of_change_10', 'high_low_spread',
    'repo_rate_change', 'usd_inr_sma_10', 'usd_inr_deviation'
]
target = 'target_return'

print("Beginning Multi-Stage Champion-Challenger Pipeline...")

for sector, df in processed_sector_data.items():
    print(f"\n==================== SECTOR: {sector.upper()} ====================")
    df['date_parsed'] = pd.to_datetime(df['date'])
    
    train_df = df[(df['date_parsed'] >= '2022-01-01') & (df['date_parsed'] <= '2024-12-31')].copy()
    val_df = df[(df['date_parsed'] >= '2025-01-01') & (df['date_parsed'] <= '2025-12-31')].copy()
    full_history_df = df[(df['date_parsed'] >= '2022-01-01') & (df['date_parsed'] <= '2025-12-31')].copy()
    
    # Isolate full inference window specified by desk parameters
    live_2026_df = df[(df['date_parsed'] >= '2026-01-01') & (df['date_parsed'] <= '2026-06-30')].copy()

    if train_df.empty or val_df.empty:
        print(f" ! Warning: Data splits missing for {sector}.")
        continue

    model_engine = SectorModelEngine(features=features, target=target, sector_name=sector)
    
    model_engine.train_and_select_champion(train_df, val_df, n_trials=10)
    model_engine.refit_on_full_history(full_history_df)
    
    if not live_2026_df.empty:
        print(" -> Generating Full 2026 Inference Log (January 1 to June 30)...")
        insights_historical_pool = model_engine.generate_ensemble_insights(live_2026_df)
        
        # Format and display the complete historical trading ledger
        pd.set_option('display.max_rows', 15)
        formatted_output = insights_historical_pool.sort_values(by=['ticker', 'date']).reset_index(drop=True)
        
        print(f"\nComplete 2026 Price Evaluation Ledger for {sector} Desk:")
        print(formatted_output[['date', 'ticker', 'actual_price', 'predicted_price']])

Beginning Multi-Stage Champion-Challenger Pipeline...

==================== SECTOR: BANKING ====================
 -> Generating Full 2026 Inference Log (January 1 to June 30)...

Complete 2026 Price Evaluation Ledger for Banking Desk:
           date ticker  actual_price  predicted_price
0    2026-01-01   HDFC    335.197876       335.936268
1    2026-01-02   HDFC    338.055817       338.462188
2    2026-01-05   HDFC    335.561005       335.895902
3    2026-01-06   HDFC    336.056549       336.330949
4    2026-01-07   HDFC    333.323486       332.803766
..          ...    ...           ...              ...
379  2026-06-23    SBI    680.477173       680.381297
380  2026-06-24    SBI    673.833313       673.571842
381  2026-06-25    SBI    665.797424       663.130512
382  2026-06-26    SBI    667.240723       665.020506
383  2026-06-29    SBI    675.646179       675.748950

[384 rows x 4 columns]

==================== SECTOR: IT ====================
 -> Generating Full 2026 Inference Log 

In [ ]:
import sys
import os
import pandas as pd

# 1. Ensure Python can find the parent project root containing the 'src' directory
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.portfolio_manager import NRIPortfolioManager

print("Initializing Motilal Oswal NRI Advisory Layer...")
portfolio_manager = NRIPortfolioManager()

# Simulate current institutional aggregate NRI holdings across the firm for our tracker
mock_aggregate_nri_holdings = {
    'SBI': 0.19,   # Near the 20% Banking statutory cap
    'HDFC': 0.04,
    'ICICI': 0.08,
    'TCS': 0.12
}

# Simulate a specific high-net-worth NRI client's personal portfolio positions
mock_client_portfolio = {
    'WIPRO': {'holding_days': 120, 'unrealized_gain_pct': 0.15},
    'TCS': {'holding_days': 410, 'unrealized_gain_pct': 0.25}
}

# 2. Extract the absolute latest 2026 snapshot rows from the historical pool
advisor_dashboard = insights_historical_pool.groupby('ticker').last().reset_index()

# FIX: Reconstruct 'predicted_next_day_return' dynamically from the price basis columns
# This satisfies the data manager contracts without altering the standalone tracking charts!
advisor_dashboard['predicted_next_day_return'] = (
    (advisor_dashboard['predicted_price'] - advisor_dashboard['actual_price']) / advisor_dashboard['actual_price']
)

print("\nStep 1: Running FEMA Sectoral Compliance Checks...")
compliance_df = portfolio_manager.filter_compliance_caps(advisor_dashboard, mock_aggregate_nri_holdings)

print("Step 2: Executing Tax-Aware Client Allocation Optimization...")
final_nri_routing = portfolio_manager.optimize_tax_exit(compliance_df, mock_client_portfolio)

print("\n================== NRI ADVISORY EXECUTION MATRIX ==================")
# Format the return column strictly for presentation visualization at the very end
final_nri_routing['predicted_next_day_return'] = (final_nri_routing['predicted_next_day_return'] * 100).round(2).astype(str) + '%'
print(final_nri_routing[['ticker', 'sector', 'actual_price', 'predicted_price', 'predicted_next_day_return', 'fema_status', 'nri_action_signal']])

Initializing Motilal Oswal NRI Advisory Layer...

Step 1: Running FEMA Sectoral Compliance Checks...
Step 2: Executing Tax-Aware Client Allocation Optimization...


KeyError: 'predicted_next_day_return'